In [54]:
import os
import sys
import warnings
import math
import pandas as pd

from joblib import parallel_backend

from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RepeatedStratifiedKFold

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier

# Élimination des avertissements
if not sys.warnoptions:
    warnings.simplefilter("ignore")
    os.environ["PYTHONWARNINGS"] = "ignore"

In [55]:
# Stopwords & punctuation
import string
from nltk.corpus import stopwords
punct = string.punctuation
stopw = stopwords.words('english') + [x for x in punct]
stopw += [x.translate
    (str.maketrans('', '', punct)) for x in stopwords.words('english')]

stopw +=  ["'d", "'ll", "'re", "'s", "'ve", '``', 'could', 'might', 'must', "n't", 'need', 'sha', 'wo', 'would']

# Tokenizer
from nltk.tokenize import word_tokenize

def tokenize_remove_stopwords(text):
 for token in word_tokenize(text):
    if token in stopw: continue
    yield (token)

**Lecture des données**

In [56]:
# Lecture du jeu de données et séparation de celles-ci en ensembles d'entraînement et de test
train = pd.read_excel('../data/training_datasets/train_dataset_40pc.xlsx')
test = pd.read_csv('../data/test_dataset_10.csv')

**Entraînement des modèles**

In [57]:
X_train, y_train = train.text_post.astype('str'), train.category
X_test, y_test = test.text_post.astype('str'), test.category

In [64]:
# Définition du pipeline
n_features = [5000, 10000, 15000]

pipeline = Pipeline(
    [
        ("vectorizer", TfidfVectorizer(            
            stop_words=stopw,
            tokenizer=word_tokenize,
            token_pattern=None)),
        ("classify", "passthrough"),
    ]
)

param_grid = [
    {
        "vectorizer__max_features": n_features,
        "classify" : [
            LogisticRegression(n_jobs=1, solver='saga'), 
            LinearSVC(dual="auto"),
            MultinomialNB(),
            ]
    }
]

In [65]:
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=0)

grid_search = GridSearchCV(
    pipeline, 
    param_grid=param_grid, 
    verbose=5, 
    cv=cv,
    refit='f1_macro', 
    scoring=['accuracy','f1_macro']
)

In [ ]:
grid_search.fit(X_train, y_train)

Fitting 50 folds for each of 9 candidates, totalling 450 fits
[CV 1/50] END classify=LogisticRegression(n_jobs=1, solver='saga'), vectorizer__max_features=5000; accuracy: (test=0.825) f1_macro: (test=0.811) total time=  22.6s
[CV 2/50] END classify=LogisticRegression(n_jobs=1, solver='saga'), vectorizer__max_features=5000; accuracy: (test=0.827) f1_macro: (test=0.813) total time=  21.0s
[CV 3/50] END classify=LogisticRegression(n_jobs=1, solver='saga'), vectorizer__max_features=5000; accuracy: (test=0.826) f1_macro: (test=0.812) total time=  20.9s
[CV 4/50] END classify=LogisticRegression(n_jobs=1, solver='saga'), vectorizer__max_features=5000; accuracy: (test=0.816) f1_macro: (test=0.801) total time=  21.3s
[CV 5/50] END classify=LogisticRegression(n_jobs=1, solver='saga'), vectorizer__max_features=5000; accuracy: (test=0.824) f1_macro: (test=0.810) total time=  22.0s
[CV 6/50] END classify=LogisticRegression(n_jobs=1, solver='saga'), vectorizer__max_features=5000; accuracy: (test=0.8

In [ ]:
grid_search.best_params_

{'classify': LogisticRegression(n_jobs=1), 'vectorizer__max_features': 5000}

In [ ]:
results_cv = pd.DataFrame(grid_search.cv_results_)
results_cv

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_classify,param_vectorizer__max_features,params,split0_test_accuracy,split1_test_accuracy,split2_test_accuracy,...,split43_test_f1_macro,split44_test_f1_macro,split45_test_f1_macro,split46_test_f1_macro,split47_test_f1_macro,split48_test_f1_macro,split49_test_f1_macro,mean_test_f1_macro,std_test_f1_macro,rank_test_f1_macro
0,18.102405,2.782544,4.441183,0.501296,LogisticRegression(n_jobs=1),5000,"{'classify': LogisticRegression(n_jobs=1), 've...",0.825,0.826875,0.82575,...,0.810105,0.810766,0.803367,0.819121,0.801817,0.809006,0.813734,0.810205,0.004094,1


In [ ]:
# results_cv = results_cv[
#     ['param_classify', 'param_vectorizer__max_features', 
#      'split0_test_accuracy', 'split1_test_accuracy', 'split2_test_accuracy',
#        'split3_test_accuracy', 'split4_test_accuracy', 'mean_test_accuracy',
#        'std_test_accuracy', 'rank_test_accuracy', 'split0_test_f1_macro',
#        'split1_test_f1_macro', 'split2_test_f1_macro', 'split3_test_f1_macro',
#        'split4_test_f1_macro', 'mean_test_f1_macro', 'std_test_f1_macro',
#        'rank_test_f1_macro']
#     ]

# results_cv.sort_values(by='rank_test_f1_macro')